# Data Leakage Prevention (DLP) ML Classifier - Training Pipeline

This Kaggle notebook fine-tunes a **Phi-2** base model using **Unsloth** and **LoRA**.
The dataset should be located in your Kaggle Inputs (in `.jsonl` format). The objective is to teach the model to classify risk based on formatted JSON features and text into exactly one of `ALLOW`, `REDACT`, `ESCALATE`, or `BLOCK`.

**Instructions**
1. Open this notebook in Kaggle (`File -> Import Notebook`).
2. Ensure you have selected a **T4 GPU x2** or single **T4/P100 GPU** accelerator.
3. Run all cells sequentially.

In [ ]:
# --- Section 1: Setup ---
# Install Unsloth and required dependencies for Kaggle environments
# Unsloth is optimized for single T4/A100 GPUs and significantly speeds up fine-tuning while reducing VRAM usage.

%%capture
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# --- Section 2: Kaggle Paths Setup ---
import os

# Configuration Variables - Adjust these if your dataset name is different
# In Kaggle, datasets are usually mounted at /kaggle/input/
DATASET_PATH = "/kaggle/input/dlp-dataset/dlp_dataset.jsonl"

# Output should go to /kaggle/working/ so it is saved when the session ends
OUTPUT_MODEL_PATH = "/kaggle/working/dlp_phi2_lora_outputs"
os.makedirs(OUTPUT_MODEL_PATH, exist_ok=True)

In [ ]:
# --- Section 3: Load Dataset ---
import json
from datasets import load_dataset

print(f"Loading dataset from: {DATASET_PATH}")
# Read directly as a huggingface Dataset
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(f"Loaded {len(dataset)} training samples.")
# Display the first sample to verify structure
for key, value in dataset[0].items():
    print(f"[{key}]:", value)

In [ ]:
# --- Section 4: Model & Tokenizer Loading ---
import torch
from unsloth import FastModel

# Model Configuration
max_seq_length = 1024 # Standard context size for short text classification
dtype = None # Auto-detect. Use Float16 on T4 V100, Bfloat16 on Ampere+
load_in_4bit = True # 4-bit quantization via bitsandbytes

print("Optimizing Phi-2 via Unsloth & BitsandBytes (4-bit)...")
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/phi-2", # Phi-2 architecture
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Phi-2 is a base model without a standard chat template, so we just use the raw tokenizer 
# padded appropriately.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# --- Section 5: Preprocessing ---
# This is our unified prompt logic explicitly instructing the model on what to output.
# We present the input text along with pre-extracted features and surface components.

def format_prompts(examples):
    """
    Batched mapping function to convert dataset rows into a raw Instruction/Output text.
    """
    surfaces = examples["surface"]
    features_list = examples["features"]
    texts = examples["text"]
    labels = examples["label"]
    
    formatted_texts = []
    
    for surface, features, text, label in zip(surfaces, features_list, texts, labels):
        # Convert dictionary to newline-separated properties
        features_str = "\n".join([f"{k}={v}" for k, v in features.items()])
        
        # Phi-2 responds simply to Instruct / Output format.
        prompt = f"""Instruct: You are a classifier for AI agents Data Leakage Prevention (DLP).

Your task is to analyze the input and classify its risk level into EXACTLY one of:
ALLOW, REDACT, ESCALATE, BLOCK.

You are given:
- SURFACE: where the data appears (OUTPUT:LLM's final output to the user, TOOL_ARGS: arguments for the tool calling, TOOL_RESULT: result from the tool)
- FEATURES: extracted signals from deterministic analysis
- TEXT: the raw content

You must follow these rules:

DEFINITIONS:
- ALLOW: Content is safe.
- REDACT: Contains limited personal data.
- ESCALATE: Content is ambiguous or potentially risky.
- BLOCK: Content clearly contains highly sensitive data or secrets. Always BLOCK if a real secret is present.

FEATURE USAGE GUIDELINES:
- num_emails > 0 -> indicates presence of PII
- num_secrets_detected > 0 -> strong signal for BLOCK
- If unsure, choose ESCALATE.

PRIORITY RULES:
1. If real secrets or credentials are present -> BLOCK
2. Else if clear PII is present -> REDACT
3. Else if ambiguous or suspicious -> ESCALATE
4. Else -> ALLOW

### Input:
[SURFACE={surface}]

[FEATURES]
{features_str}

[TEXT]
{text}
Output: {label}"""
        
        formatted_texts.append(prompt)
        
    return {"formatted_text": formatted_texts}

# Map and format the dataset instances right before training
print("Formatting the dataset instances...")
dataset = dataset.map(format_prompts, batched=True)
print("Sample Prompt Input:")
print("===============")
print(dataset[0]["formatted_text"])
print("===============")

In [ ]:
# --- Section 6: Apply LoRA Adapters ---

# Instead of updating billions of parameters, LoRA only updates ~1% 
# For Phi-2, Unsloth automatically targets the right projection matrices. 
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, 
    finetune_language_layers   = True,  
    finetune_attention_modules = True,  
    finetune_mlp_modules       = True,  
    r = 16, # Rank mapping size (16 is a solid default)
    lora_alpha = 16,
    lora_dropout = 0, # Optimized: Keep at 0
    bias = "none",    # Optimized: Keep at "none"
    use_gradient_checkpointing = "unsloth", # Crucial for limits
    random_state = 3407,
)

print(f"PEFT / LoRA modules injected!")
model.print_trainable_parameters()

In [ ]:
# --- Section 7: Training Setup ---
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import torch

# Initialize HF standard Trainer via PEFT/Unsloth integrations
# Configure hyper-params tailored for T4 VRAM.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "formatted_text",# Point to our dynamically built strings
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        per_device_train_batch_size = 1,  
        gradient_accumulation_steps = 8,  
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not FastModel.is_bfloat16_supported(),
        bf16 = FastModel.is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Apply response-only training to ignore user prompt from loss calculation
# For our format, we appended "Output: {label}"
trainer = train_on_responses_only(
    trainer,
    instruction_part = "\nOutput: ",
    response_part = "", # Not strictly needed if instruction_part is simply a prefix
)

# Benchmark memory footprint before training:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name} | Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved currently.")

In [ ]:
# --- Section 8: Execute Training Process ---
print("Initiating full training loop over 6000 samples...")
trainer_stats = trainer.train()

print(f"\nFinal Training loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# --- Section 9: Save Weights ---
print(f"Persisting model adapters and tokenizer weights to Kaggle Working Directory...")
print(f"Path: {OUTPUT_MODEL_PATH}")

model.save_pretrained(OUTPUT_MODEL_PATH)      # Saves adapter_config.json and adapter_model.bin/safetensors
tokenizer.save_pretrained(OUTPUT_MODEL_PATH)  # Saves tokenizer_config.json

print("Save completed successfully. You can verify it in the output section at the right side of your Kaggle notebook.")

In [ ]:
# --- Section 10: Export and Download zip ---
import shutil
import os

print("Zipping the LoRA adapter folder locally for direct download...")
ARCHIVE_NAME = "/kaggle/working/dlp_model_lora"

# Creates /kaggle/working/dlp_model_lora.zip
shutil.make_archive(ARCHIVE_NAME, 'zip', OUTPUT_MODEL_PATH)

print(f"Archive created at {ARCHIVE_NAME}.zip.")
print("To download, click on the three dots next to the file in your Kaggle output directory.")
print("Finished!")